<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/09b_goal_directed_generation.ipynb)

# 09b · 目标导向的分子逆向设计：迁移微调、代理模型欺骗与可合成性评估

在 `08a` 与 `09a` 中，我们分别掌握了基于自回归语言模型和基于连续潜空间的无条件分子生成。然而在真实的材料研发中，科学家真正关心的问题是**目标导向的逆向设计（Goal-Directed Inverse Design）**：如何让生成模型主动朝着“更高光电转换效率（PCE）”、“更宽带隙”或“更高溶解度”的方向演化？

要实现这一闭环，我们需要把前几讲的核心工具串联成一条完整的管线，同时直面生成式材料设计中最隐蔽、最普遍的两大工程陷阱：

1. **构建快速“裁判”：代理预测模型（Surrogate Model）**：量子化学 DFT 计算一个有机光伏分子的 PCE 需要数小时，无法在生成循环中实时打分；我们首先复用第 02 讲的 `Morgan 指纹 + 随机森林` 训练一个毫秒级响应的 PCE 代理裁判；
2. **分布定向推移：迭代精英微调（Iterative Fine-Tuning）**：从 `08a` 预训练好的通用有机光伏语言模型出发，在 Top-10% 高 PCE 分子子集上以小学习率微调，观察生成分子的 PCE 分布右移，并验证模型如何在没有任何化学先验指导的情况下“自主重新发现”**苯并噻二唑（Benzothiadiazole）**等经典吸电子受体单元（内嵌 **练习 09b-1**）；
3. **头号陷阱：代理模型欺骗（Surrogate Model Exploitation / Reward Hacking）**：当优化器或生成模型过度追求代理模型的高分时，极易钻进代理模型的适用域盲区——我们将现场演示为什么随机森林会给**一个水分子**打出 `1.2%` 的光伏效率且各棵决策树高度一致，为什么二值指纹会让无限拉长的聚合物得分饱和，以及如何用 Tanimoto 适用域约束与悲观下界（LCB）防御模型欺骗（内嵌 **练习 09b-2** 与 **练习 09b-3**）；
4. **二号陷阱：合成可及性（SA Score）与候选材料诚实检验清单**。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "09 - Generative Models"))
    get_ipython().run_line_magic("pip", "install -q rdkit")

In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import re
import sys
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Draw, RDConfig, rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

<!-- ai4m-style -->
> **课程统一绘图风格**：下方单元加载 `ai4m.mplstyle`。

In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))
import ai4m_style as ai4m

C = ai4m.use()

In [ ]:
DATA_DIR = "data" if os.path.exists("data/cep.csv") else "course/09 - Generative Models/data"
df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print(f"Loaded CEP dataset: {len(df):,} molecules")
df.head(3)

---
## 1. 构建快速“裁判”：预测光电转换效率（PCE）的随机森林代理模型

在哈佛清洁能源项目（CEP）中，每个分子的 `PCE`（Power Conversion Efficiency，单位 `%`）是由量子化学密度泛函理论（DFT）结合 Scharber 模型估算得到的。由于无法在每次采样生成新分子时实时运行数小时的 DFT 计算，我们在整个数据集上提取 2048 位二值 Morgan 指纹（半径 $r = 2$），留出 20% 作为测试集，训练一个包含 200 棵决策树的随机森林回归器作为**代理模型（Surrogate Oracle）**。

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = np.array([fp_gen.GetFingerprintAsNumPy(m) for m in mols], dtype=np.uint8)
y = df["PCE"].values

train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(fingerprints[train_idx], y[train_idx])

pred_test = rf.predict(fingerprints[test_idx])
rmse = np.sqrt(np.mean((pred_test - y[test_idx]) ** 2))
r2 = 1.0 - np.sum((pred_test - y[test_idx]) ** 2) / np.sum((y[test_idx] - y[test_idx].mean()) ** 2)
print(f"Surrogate Random Forest | Test RMSE = {rmse:.2f}%, R^2 = {r2:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0))
axes[0].scatter(y[test_idx], pred_test, s=3, alpha=0.3)
axes[0].plot([-7, 12], [-7, 12], "k--", lw=1)
axes[0].set_xlabel("True DFT PCE (%)")
axes[0].set_ylabel("Predicted PCE (%)")
axes[0].set_title(f"Surrogate Model on Test Set (R² = {r2:.2f})")

axes[1].hist(y, bins=60, alpha=0.85)
axes[1].set_xlabel("DFT PCE (%)")
axes[1].set_ylabel("Count")
axes[1].set_title("PCE Distribution in CEP Dataset")
plt.tight_layout()
plt.show()
print(f"CEP PCE Statistics: Mean = {y.mean():.2f}%, Median = {np.median(y):.2f}%, Max = {y.max():.2f}%")

代理模型在同分布测试集上的决定系数达到 $R^2 \approx 0.74$，足以作为快速初筛工具；但请牢记它的测试集均方根误差 $\text{RMSE} \approx 1.3\%$——**代理模型只是真实物理性质的有噪近似，而且它的准确性严格依赖于输入分子落在训练数据的适用域（Applicability Domain）之内**。

---
## 2. 目标导向微调：把生成分布推向高 PCE 区域

要让生成模型偏爱高 PCE 分子，工业界有两条主流技术路线：
1. **策略梯度强化学习（如 REINVENT）**：将代理模型预测的 PCE 作为奖励信号（Reward），通过策略梯度直接最大化期望奖励；
2. **精英子集迁移微调（Transfer Fine-Tuning / Hill-Climb MLE）**：先在全量数据上预训练先验语言模型 $P_{\text{prior}}(x)$ 掌握分子语法，再筛选出性质最优的 Top-$k\%$ 分子子集，用较小的学习率继续微调几轮。

相比于容易发生模式坍缩（Mode Collapse）的强化学习，精英子集微调无需复杂的超参数调节，且天然受真实精英分子的序列分布约束。下面我们复用 `08a` 中预训练好的 `SmilesRNN` 架构与 `data/cep_rnn.pt` 权重作为先验模型 `prior_model`。

In [ ]:
TOKEN_PATTERN = re.compile(r"(\[[^\]]+\]|Br|Cl|[BCNOPSFIbcnops]|\(|\)|\.|=|#|-|\+|\\|/|:|%\d{2}|\d)")

def tokenize(smiles):
    tokens = TOKEN_PATTERN.findall(smiles)
    assert "".join(tokens) == smiles, smiles
    return tokens

token_lists = [tokenize(s) for s in df["smiles"]]
vocab = ["<pad>", "^", "$"] + sorted({t for ts in token_lists for t in ts})
stoi = {t: i for i, t in enumerate(vocab)}
max_len = max(len(ts) for ts in token_lists) + 2

data = torch.zeros(len(token_lists), max_len, dtype=torch.long)
for i, tokens in enumerate(token_lists):
    ids = [stoi["^"]] + [stoi[t] for t in tokens] + [stoi["$"]]
    data[i, : len(ids)] = torch.tensor(ids)

class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, n_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, state=None):
        out, state = self.lstm(self.embedding(x), state)
        return self.linear(out), state

@torch.no_grad()
def sample(model, n=500, temperature=1.0, max_steps=max_len):
    model.eval()
    x = torch.full((n, 1), stoi["^"], dtype=torch.long)
    state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_steps):
        logits, state = model(x, state)
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)
        out = torch.cat([out, x], dim=1)
        finished |= (x[:, 0] == stoi["$"])
        if finished.all():
            break

    smiles_list = []
    for row in out.tolist():
        tokens = []
        for t in row:
            if t == stoi["$"]:
                break
            tokens.append(vocab[t])
        smiles_list.append("".join(tokens))
    return smiles_list

CKPT = os.path.join(DATA_DIR, "cep_rnn.pt")
prior_model = SmilesRNN(len(vocab))
raw_ckpt = torch.load(CKPT, map_location="cpu")
prior_model.load_state_dict(raw_ckpt["model_state_dict"] if isinstance(raw_ckpt, dict) and "model_state_dict" in raw_ckpt else raw_ckpt)
prior_model.eval()
print(f"Loaded prior SmilesRNN from {CKPT} (vocab size = {len(vocab)}, max_len = {max_len})")

### 2.1 筛选 Top-10% 高 PCE 精英子集并执行微调

在挑选微调子集时，比例 $k\%$ 存在鲜明的统计权衡：
- 若阈值过于苛刻（如仅取 Top-1%，约 300 个分子），样本量过小会导致模型在几轮内死记硬背，生成分子的唯一性（Uniqueness）和新颖性下降；
- 若阈值过于宽松（如 Top-50%），子集分布与原分布几乎没有拉开差距，微调推动力太弱。

我们先选取 **Top-10%（2,997 个高 PCE 分子，PCE 门槛 $\ge 6.48\%$，均值 $7.60\%$）**，以比预训练小 10 倍的学习率（`lr = 1e-4`）微调 4 个 Epoch。

In [ ]:
top_fraction = 0.10
n_top = int(len(df) * top_fraction)
top_idx = np.argsort(-y)[:n_top]
print(f"Fine-tuning elite subset: {n_top:,} molecules | Min PCE = {y[top_idx].min():.2f}% | Mean PCE = {y[top_idx].mean():.2f}%")

tuned_model = copy.deepcopy(prior_model)
optimizer = torch.optim.Adam(tuned_model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

subset = data[top_idx]
tuned_model.train()
for epoch in range(4):
    order = np.random.permutation(len(subset))
    total_loss = 0.0
    for i in range(0, len(order), 128):
        batch = subset[order[i : i + 128]]
        logits, _ = tuned_model(batch[:, :-1])
        loss = loss_fn(logits.reshape(-1, len(vocab)), batch[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(tuned_model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
    print(f"Fine-tune Epoch {epoch + 1}/4 | Cross-Entropy Loss = {total_loss / (len(order) / 128):.4f}")
tuned_model.eval();

### 2.2 微调前后的性质分布推移与电子给体-受体构筑单元演化

我们从微调前的 `prior_model` 与微调后的 `tuned_model` 各采样 1,500 个分子，用随机森林代理模型评估其预测 PCE，并统计各个杂环构筑单元（Building Blocks）在微调前后的出现频率变化。

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

def analyse_generator(model, name, n=1500, temperature=1.0):
    samples = sample(model, n=n, temperature=temperature)
    parsed = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m, canonical=True) for m in parsed if m is not None and m.GetNumAtoms() > 0]
    unique = sorted(set(valid))
    fps = np.array([fp_gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(c)) for c in unique], dtype=np.uint8)
    predictions = rf.predict(fps)
    print(
        f"{name:14s} | Valid: {len(valid)/n:.3f} | Unique: {len(unique)/max(len(valid),1):.3f} | "
        f"Novel: {len(set(unique)-canonical_train)/max(len(unique),1):.3f} | "
        f"Mean Pred PCE: {predictions.mean():.2f}% (Max: {predictions.max():.2f}%)"
    )
    return pd.DataFrame({"smiles": unique, "pred_PCE": predictions})

before = analyse_generator(prior_model, "Before Tuning")
after = analyse_generator(tuned_model, "After Tuning")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.0))
bins = np.linspace(-2, 12, 50)
ax.hist(y, bins=bins, density=True, alpha=0.35, label="CEP Training Data (True DFT)")
ax.hist(before["pred_PCE"], bins=bins, density=True, histtype="step", lw=2, label="Generated Before Fine-Tuning")
ax.hist(after["pred_PCE"], bins=bins, density=True, histtype="step", lw=2, label="Generated After Fine-Tuning")
ax.axvline(y[top_idx].min(), color="gray", ls=":", lw=1.5, label="Top-10% Elite Threshold")
ax.set_xlabel("PCE (%)")
ax.set_ylabel("Density")
ax.set_title("Goal-Directed Fine-Tuning Shifts the Generated PCE Distribution")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
best_candidates = after.nlargest(8, "pred_PCE")
Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in best_candidates["smiles"]],
    molsPerRow=4,
    subImgSize=(270, 210),
    legends=[f"Pred PCE: {p:.2f}%" for p in best_candidates["pred_PCE"]],
)

In [ ]:
# 统计微调前后各共轭构筑单元（杂环片段）的出现频率变化
fragments = {
    "thiadiazole (acceptor)": "c1nsnc1",
    "thiophene (donor)": "c1ccsc1",
    "furan": "c1ccoc1",
    "pyrrole": "c1cc[nH]c1",
    "benzene": "c1ccccc1",
    "selenophene": "c1cc[se]c1",
}

def fragment_frequency(smiles_list):
    parsed = [Chem.MolFromSmiles(s) for s in smiles_list]
    return {
        name: np.mean([m.HasSubstructMatch(Chem.MolFromSmiles(sm)) for m in parsed])
        for name, sm in fragments.items()
    }

freq_df = pd.DataFrame({
    "CEP All": fragment_frequency(df["smiles"].sample(2000, random_state=0)),
    "CEP Top 10%": fragment_frequency(df["smiles"].values[top_idx][:2000]),
    "Generated Before": fragment_frequency(before["smiles"]),
    "Generated After": fragment_frequency(after["smiles"]),
}).round(3)

fig, ax = plt.subplots(figsize=(7.2, 3.6))
freq_df.plot.barh(ax=ax)
ax.set_xlabel("Fraction of Molecules Containing Fragment")
ax.set_title("Fine-Tuning Autonomously Enriches Electron-Accepting Thiadiazole Units")
plt.tight_layout()
plt.show()

freq_df

观察上方表格与条形图中的 `thiadiazole (acceptor)`（噻二唑，即苯并噻二唑 `[2,1,3]-benzothiadiazole` 的核心杂环）：
- 在全体 CEP 数据库中，仅约 27% 的分子含有噻二唑单元；而在 Top-10% 高效光伏分子中，这一比例高达 **87%**；
- 经过短短 4 轮微调，生成模型在完全没有被告知“给体-受体（Donor-Acceptor, D-A）推拉电子能带工程理论”的前提下，自动将生成分子中噻二唑单元的占比从 `26%` 大幅提升至 **`67%`**，同时降低了富电子呋喃/吡咯的占比！这生动展示了序列生成模型如何从高维结构数据中提炼并复现领域专家的分子设计法则。

### 练习 09b-1：精英子集比例的权衡实验（Top 1% vs. 10% vs. 30%）

在第 2.1 节中我们指出：微调子集选得太窄会导致模型“死记硬背（过拟合）”，选得太宽则缺乏定向推动力。

请在下方编写代码，每次从 `prior_model` 深拷贝一份新模型（`copy.deepcopy(prior_model)`），分别在 PCE 最高的 **1%（约 300 个分子）**、**10%（约 3,000 个分子）**与 **30%（约 9,000 个分子）**子集上以相同步数（或相同迭代次数）微调 4 轮，各采样 800 个分子，对比三组模型的：
1. 唯一性（`Uniqueness`）与新颖性（`Novelty`）；
2. 代理模型预测的平均 PCE（`Mean Predicted PCE`）。

In [ ]:
# 练习 09b-1：在此对比 Top 1%, 10%, 30% 微调子集的生成指标与预测 PCE
def evaluate_finetune_fractions(fractions=(0.01, 0.10, 0.30)):
    # TODO: 对每个 fraction 从 prior_model 深拷贝并微调 4 轮，调用 analyse_generator 汇总对比
    pass


---
## 3. 头号陷阱：代理模型欺骗（Model Exploitation）与适用域崩塌

### 3.1 检验微调分子是否仍在代理模型的适用域（Applicability Domain）内

在宣布“微调后预测 PCE 提升了近 2 个百分点”之前，一个严谨的研究者必须先追问：**这些新生成的分子还在随机森林代理模型见过的数据分布之内吗？代理模型给它们打出的高分可信吗？**

我们复用 `03b` 的方法，计算每个微调生成分子到 CEP 训练集参考样本的**最近邻 Tanimoto 相似度（Max Tanimoto Similarity）**，并与训练集内部真实分子彼此之间的最近邻相似度分布进行对照。

In [ ]:
reference_idx = np.random.RandomState(0).choice(len(df), 3000, replace=False)
reference_fps = [fp_gen.GetFingerprint(mols[i]) for i in reference_idx]

def max_similarity_to_ref(smiles_list):
    out = []
    for s in smiles_list:
        fp = fp_gen.GetFingerprint(Chem.MolFromSmiles(s))
        out.append(max(DataStructs.BulkTanimotoSimilarity(fp, reference_fps)))
    return np.array(out)

sim_before = max_similarity_to_ref(before["smiles"][:500])
sim_after = max_similarity_to_ref(after["smiles"][:500])

# 对照基准：CEP 训练集分子与其余训练集分子之间的次近邻相似度（排除自身 1.0）
inside_sim = []
for s in df["smiles"].sample(500, random_state=1):
    fp = fp_gen.GetFingerprint(Chem.MolFromSmiles(s))
    sims = sorted(DataStructs.BulkTanimotoSimilarity(fp, reference_fps))
    inside_sim.append(sims[-2])
inside_sim = np.array(inside_sim)

print(f"Median Nearest-Neighbor Tanimoto Similarity:")
print(f"  Within CEP Training Set : {np.median(inside_sim):.3f}")
print(f"  Generated (Before Tune) : {np.median(sim_before):.3f}")
print(f"  Generated (After Tune)  : {np.median(sim_after):.3f}")

fig, ax = plt.subplots(figsize=(6.2, 3.6))
bins = np.linspace(0.2, 1.0, 40)
ax.hist(inside_sim, bins=bins, density=True, alpha=0.5, label="CEP Molecule vs. CEP")
ax.hist(sim_after, bins=bins, density=True, histtype="step", lw=2, label="Fine-Tuned Generated vs. CEP")
ax.set_xlabel("Max Tanimoto Similarity to Training Set")
ax.set_ylabel("Density")
ax.set_title("Fine-Tuned Molecules Remain Inside the Applicability Domain")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

可以看到，微调后的生成分子与训练集的最近邻相似度中位数（约 `0.48`）同训练集内部水平（约 `0.49`）几乎完全一致。这是因为小学习率的语言模型微调始终受真实分子的序列似然牵引，不会轻易脱离训练流形。

### 3.2 探针实验：当优化器离开适用域时，代理模型会错得多离谱？

如果不用温和的微调，而是把代理模型交给一个不受约束的搜索算法（如遗传算法、爬山法或无 KL 约束的强化学习）去全力最大化预测 PCE，会发生什么？

这种现象在机器学习中被称为**代理模型欺骗（Model Exploitation / Reward Hacking）**：搜索算法很快会发现代理模型的数学盲区，构造出在物理上荒谬绝伦、但代理模型却自信满满给出高分的“对抗分子”。

下面我们构造一组极端探针分子——包括不同长度的苯并噻二唑齐聚物（$\times 1, \times 3, \times 6, \times 12$）、完全没有 $\pi$ 共轭体系的正辛烷（`CCCCCCCC`）、四十碳长链烷烃（`C40`）以及**一个普通的水分子（`O`）**——直接送入随机森林代理模型，同时记录 200 棵决策树预测值的标准差（`Tree Spread`，常被用作集成不确定度指标）以及最近邻 Tanimoto 相似度。

In [ ]:
benzothiadiazole = "c1ccc2nsnc2c1"
probes = {
    "Benzothiadiazole x1": benzothiadiazole,
    "Benzothiadiazole x3": "-".join([benzothiadiazole] * 3),
    "Benzothiadiazole x6": "-".join([benzothiadiazole] * 6),
    "Benzothiadiazole x12": "-".join([benzothiadiazole] * 12),
    "Thiophene x12": "-".join(["c1ccsc1"] * 12),
    "Benzene x20": "-".join(["c1ccccc1"] * 20),
    "n-Octane (No conjugation)": "CCCCCCCC",
    "Alkane C40": "C" * 40,
    "Water (H2O)": "O",
}

probe_rows = []
for name, smi in probes.items():
    mol = Chem.MolFromSmiles(smi)
    x_fp = np.array([fp_gen.GetFingerprintAsNumPy(mol)], dtype=np.uint8)
    tree_preds = np.array([t.predict(x_fp)[0] for t in rf.estimators_])
    probe_rows.append({
        "Probe Molecule": name,
        "Heavy Atoms": mol.GetNumHeavyAtoms(),
        "Predicted PCE (%)": round(float(tree_preds.mean()), 2),
        "Tree Spread Std (%)": round(float(tree_preds.std()), 2),
        "Max Tanimoto Sim": round(max(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mol), reference_fps)), 3),
    })

typical_spread = np.array([t.predict(fingerprints[test_idx[:300]]) for t in rf.estimators_]).std(axis=0).mean()
print(f"Reference | Mean Tree Spread on CEP Test Set = {typical_spread:.2f}%")
print(f"Reference | Heavy Atom Range in CEP Training = {min(m.GetNumHeavyAtoms() for m in mols)} - {max(m.GetNumHeavyAtoms() for m in mols)}")
pd.DataFrame(probe_rows).set_index("Probe Molecule")

仔细审视上方这张表，它揭示了材料逆向设计中三个必须刻在脑海里的核心教训：

1. **回归模型永远不会主动拒绝回答，哪怕输入毫无物理意义**：
   水分子（`H2O`）和正辛烷（`CCCCCCCC`）连一根共轭双键都没有，根本不可能作为有机太阳能电池的吸光给体，但随机森林依然面不改色地给水分子预测出 `1.20%` 的光电转换效率，给长链烷烃预测出 `1.5%` 以上的效率。
2. **集成模型的内部方差（Tree Spread）在分布外可能彻底失灵**：
   人们常以为“离开训练分布后，随机森林各棵树的分歧一定会变大”。但看表格最后三行：水分子和烷烃的 `Tree Spread`（约 `0.46%`）甚至**比正常测试集分子的平均分歧（`1.07%`）还要小一半**！原因很简单：水分子的 2048 位 Morgan 指纹几乎全为 `0`，在全部 200 棵决策树中，遇到“特征全为 0”的分支都会一致地走入同一片冷门叶节点，从而**步调高度一致地输出同一个错误答案**。
3. **二值指纹的比特饱和（Bit Saturation）使模型对重复单元失明**：
   对比 `Benzothiadiazole x6`（54 个重原子）与 `Benzothiadiazole x12`（108 个重原子，远超训练集上限 35 个重原子）：两者的预测 PCE 和树标准差**精确到小数点后每一位都完全相等（`6.12%`）**！这是因为二值 Morgan 指纹只记录“某种半径 $r \le 2$ 的局部子结构是否出现过（0 或 1）”——从第 6 个重复单元到第 12 个重复单元，没有任何新的局部半径子结构诞生，两者的 2048 维输入向量一模一样。
4. **真正可靠的防线是化学空间距离（Max Tanimoto Similarity）**：
   水分子（`0.000`）与烷烃（`0.030`）的最近邻 Tanimoto 相似度瞬间暴露了它们处于适用域之外。

### 练习 09b-2：无约束 vs. 适用域约束的片段爬山搜索（Hill Climbing）

现在我们写一个自动优化器，体验当优化器直接最大化代理模型得分时是如何一步步走向模型欺骗的，以及如何用我们在 `04a`/`04b` 学过的**悲观置信下界（Lower Confidence Bound, $\hat{\mu} - 2\hat{\sigma}$）**与**适用域硬阈值**将其拉回正轨。

设每个分子由若干个共轭芳香片段通过单键 `-` 串联而成：
```python
FRAGMENTS = [
    "c1ccsc1", "c1ccoc1", "c1cc[nH]c1", "c1ccccc1",
    "c1cc[se]c1", "c1ccc2nsnc2c1", "c1ccc2[nH]ccc2c1",
]
```
从初始三聚体 `["c1ccsc1", "c1ccc2nsnc2c1", "c1ccsc1"]` 出发，每一步随机执行**插入、删除或替换**一个片段，运行 120 步爬山搜索，对比以下两种策略找到的最终分子：
1. **无约束策略（Unconstrained）**：直接最大化代理模型预测均值 $\hat{\mu}(\mathbf{x})$；
2. **受约束策略（Constrained）**：若候选分子到训练集的最近邻 Tanimoto 相似度 `< 0.40` 则直接拒绝，且目标函数改为最大化悲观得分 $\hat{\mu}(\mathbf{x}) - 2\,\hat{\sigma}_{\text{trees}}(\mathbf{x})$。

In [ ]:
# 练习 09b-2：在此实现无约束 vs. 受约束的片段爬山优化器并对比终点分子
FRAGMENTS = [
    "c1ccsc1", "c1ccoc1", "c1cc[nH]c1", "c1ccccc1",
    "c1cc[se]c1", "c1ccc2nsnc2c1", "c1ccc2[nH]ccc2c1",
]

# TODO: 实现 hill_climb(constrained=False/True) 并打印两者的预测均值、悲观得分、相似度与重原子数


### 练习 09b-3：检验树模型的外推天花板——它能发现比训练集更好的材料吗？

在 `02a` 中我们学过：决策树与随机森林的预测值是叶节点内训练样本标签的加权平均，因此在数学上**永远不可能输出高于训练集最大标签的预测值**。这一性质对“生成 + 代理筛选”意味着什么？

请在下方完成一个留出检验（Hold-out Extrapolation Test）：
1. 仅使用 `PCE < 6.0%` 的分子训练一个新的随机森林 `rf_low`；
2. 用 `rf_low` 去预测那些真实 `PCE > 8.0%` 的顶尖分子（模拟我们在只见过中低效材料时，试图用代理模型去筛选突破性材料）；
3. 绘制真实 PCE vs. 预测 PCE 散点图，并统计预测值超过 `6.0%` 的样本比例（思考：为什么该比例严格为 `0.0%`？这说明在主动材料发现中为什么必须引入 DFT/实验闭环更新训练集？）。

In [ ]:
# 练习 09b-3：在此验证仅在 PCE < 6% 上训练的随机森林能否识别出 PCE > 8% 的顶尖分子
# TODO: 训练 rf_low 并评估其在 y > 8 样本上的预测上限与平均低估幅度


---
## 4. 二号陷阱：合成可及性（SA Score）与候选材料六步核查清单

即使一个生成的候选分子通过了适用域检验、且预测性能优异，走向实验室合成前还有最后一道关卡：**这个分子在有机化学上容易合成吗？**

RDKit 的 `Contrib/SA_Score` 模块提供了经典的经验**合成可及性评分（Synthetic Accessibility Score, SA Score；Ertl & Schuffenhauer, 2009）**，范围从 `1.0`（极易合成，如常见商业试剂）到 `10.0`（极难合成，含大量罕见桥环、螺环或手性中心）。

In [ ]:
sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
import sascorer

def calc_sa_scores(smiles_list):
    return np.array([sascorer.calculateScore(Chem.MolFromSmiles(s)) for s in smiles_list])

sa_train = calc_sa_scores(df["smiles"].sample(1000, random_state=0))
sa_after = calc_sa_scores(after["smiles"][:1000])
sa_best = calc_sa_scores(best_candidates["smiles"])

fig, ax = plt.subplots(figsize=(6.2, 3.5))
bins = np.linspace(1, 7, 40)
ax.hist(sa_train, bins=bins, density=True, alpha=0.5, label="CEP Training Data")
ax.hist(sa_after, bins=bins, density=True, histtype="step", lw=2, label="Generated (Fine-Tuned)")
ax.set_xlabel("SA Score (1 = Easy to Synthesize, 10 = Very Difficult)")
ax.set_ylabel("Density")
ax.set_title("Synthetic Accessibility Distribution")
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

print(
    f"Median SA Score | CEP Training: {np.median(sa_train):.2f} | "
    f"Fine-Tuned Generated: {np.median(sa_after):.2f} | Top-8 Best PCE: {np.median(sa_best):.2f}"
)

微调生成的分子在 SA Score 分布上与 CEP 训练集基本一致（中位数约 `3.4`）。但需要注意两点：
1. CEP 数据库本身是通过组合化学规则在计算机中枚举生成的虚拟库，其中相当一部分稠环异构体在实验室中并未真正合成过；
2. `SA Score` 仅统计子结构片段在历史化合物库中的出现频率，并不做真正的反应机理推演——判断一条具体合成路线是否可行，还需结合我们在 `08c` 学习的化学反应预测与逆合成规划工具。

---
## 5. 本节小结：生成式材料发现的六步诚实核查清单

在实际科研课题中，当你使用生成模型产出一批候选分子准备提交给计算或实验团队之前，请对照以下清单逐项核查：

| 核查步骤 | 具体检验方法 | 对应课程章节 |
| :--- | :--- | :--- |
| **1. 语法与价态合法性** | `Chem.MolFromSmiles` 与 `DetectChemistryProblems` 能否无错通过？ | `08a`、`09a` |
| **2. 结构分布合理性** | 芳香环数、重原子数、$sp^3$ 碳比例与关键官能团频率是否符合领域物理常识？（警惕先验空洞！） | `08a`、`09a` |
| **3. 代理模型适用域** | 候选分子到训练集的最近邻 Tanimoto 相似度是否高于安全阈值（如 $\ge 0.4$）？特征是否发生二值饱和？ | `03b`、`09b` |
| **4. 预测不确定度校准** | 结合集成模型方差或高斯过程不确定度，按悲观下界（$\hat{\mu} - k\hat{\sigma}$）或期望改进（EI）排序后是否依然领先？ | `04a`、`04b`、`09b` |
| **5. 合成可及性评估** | `SA Score` 是否处于合理区间？逆合成模型或合成化学家能否给出可行的偶联/成环路线？ | `08c`、`09b` |
| **6. 物理计算与实验闭环** | 将筛选出的 Top-$K$ 分子送入真实 DFT 计算或湿实验测试，并将新数据并入训练集迭代更新代理模型 | `03c`、`04b`、`09b` |

到目前为止，我们在 `09a` 与 `09b` 中讨论的生成模型都聚焦于一维分子序列。在材料科学的另一半版图——**材料显微表征（SEM/TEM）与二维/三维多相微结构设计**中，生成模型同样正在引发革命。下一节 **`09c · 材料显微图像超分辨率与去噪扩散概率模型（DDPM）入门`** 将带你从 `07b` 的卷积神经网络出发，亲手构建电镜超分辨率网络与条件扩散生成模型。